# Benchmarking Forecast Pipelines: Champion-Challenger Evaluation
# 预测管道基准测试：冠军-挑战者评估

Scenario: before deployment, a team compares baseline and candidate pipelines on a holdout window using business-facing metrics.

场景：上线前，团队在留出窗口上使用业务指标比较基线和候选管道。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
import time
import numpy as np
from PipelineTS.pipeline import ModelPipeline
from PipelineTS.evaluation import ModelComparison
from PipelineTS.metrics import mape, smape, picp, pinaw, winkler_score

data = make_retail_demand(n_days=240, n_stores=1).drop(columns=["store_id"])
train, valid = data.iloc[:-28].copy(), data.iloc[-28:].copy()
horizon = len(valid)

In [ ]:
candidates = {
    "baseline_fast": ["multi_output_model", "multi_step_model"],
    "forest_ensemble": ["random_forest", "extra_forest"],
    "mixed_light": ["random_forest", "multi_output_model", "regressor_chain"],
}

results = {}
timings = {}
for label, models in candidates.items():
    t0 = time.time()
    pipe = ModelPipeline(
        time_col="date",
        target_col="sales",
        lags=14,
        include_models=models,
        quantile=0.9,
        cv=2,
        random_forest__n_estimators=80,
        extra_forest__n_estimators=80,
    )
    pipe.fit(train, valid_data=valid)
    pred = pipe.predict(horizon)
    results[label] = pred
    timings[label] = time.time() - t0

timings

In [ ]:
comp = ModelComparison(time_col="date", target_col="sales")
y_true = valid["sales"].values

for name, pred in results.items():
    comp.add_result(
        name,
        y_true,
        pred["sales"].values[:horizon],
        lower=pred.get("sales_lower", None),
        upper=pred.get("sales_upper", None),
    )

table = comp.fit(
    metrics={"MAPE": mape, "sMAPE": smape},
    interval_metrics={"PICP": picp, "PINAW": pinaw, "Winkler90": lambda y, lo, hi: winkler_score(y, lo, hi, alpha=0.1)},
)
table["fit_predict_seconds"] = table["model"].map(timings)
table.sort_values("MAPE")

In [ ]:
comp.rank("MAPE")

In [ ]:
comp.plot_bar(metric_cols=["MAPE", "sMAPE", "fit_predict_seconds"])
comp.plot_predictions(time_index=valid["date"].values)